# Import Modules

In [1]:
import importlib
import os
import sys

import joblib
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
os.chdir("../../../")
sys.path.insert(0, os.getcwd())

In [27]:
from morai.experience import charters, experience
from morai.forecast import metrics, preprocessors
from morai.models import neural
from morai.utils import custom_logger, helpers

In [7]:
logger = custom_logger.setup_logging(__name__)

In [8]:
# update log level if wanting more logging
custom_logger.set_log_level("INFO")

In [9]:
pd.options.display.float_format = "{:,.2f}".format

In [10]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Data

In [11]:
pl_parquet_path = r"files/dataset/mortality_grouped.parquet"

In [12]:
# reading in the dataset
# `enable_string_cache` helps with categorical type values
pl.enable_string_cache()
lzdf = pl.scan_parquet(
    pl_parquet_path,
)

In [13]:
initial_row_count = lzdf.select(pl.len()).collect().item()
print(
    f"row count: {initial_row_count:,} \n"
    f"exposures: {lzdf.select([pl.col('amount_exposed').sum()]).collect()[0,0]:,}"
)

row count: 1,793,371 
exposures: 9,824,025,879,572.559


In [14]:
grouped_df = lzdf.collect()

In [15]:
grouped_df = grouped_df.to_pandas()

# Preparing Data

## Filter

In [16]:
model_data = grouped_df[
    (grouped_df["attained_age"] >= 50)
    # & (grouped_df["attained_age"] <= 95)
    & (grouped_df["issue_age"] >= 30)
    & (grouped_df["issue_age"] <= 80)
].copy()
model_data = model_data.reset_index(drop=True)

In [17]:
del grouped_df

## Calculated Fields

In [18]:
model_data["capped_duration"] = model_data["duration"].clip(upper=26)
model_data["qx_log_raw"] = np.log(model_data["qx_raw"] + 1)
binned_face_dict = {
    "01: 0 - 9,999": "01: 0 - 24,999",
    "02: 10,000 - 24,999": "01: 0 - 24,999",
    "03: 25,000 - 49,999": "02: 25,000 - 99,999",
    "04: 50,000 - 99,999": "02: 25,000 - 99,999",
    "05: 100,000 - 249,999": "03: 100,000 - 249,999",
    "06: 250,000 - 499,999": "04: 250,000 - 4,999,999",
    "07: 500,000 - 999,999": "04: 250,000 - 4,999,999",
    "08: 1,000,000 - 2,499,999": "04: 250,000 - 4,999,999",
    "09: 2,500,000 - 4,999,999": "04: 250,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999": "05: 5,000,000+",
    "11: 10,000,000+": "05: 5,000,000+",
}
model_data["binned_face"] = model_data["face_amount_band"].map(binned_face_dict)
model_data["binned_face"] = model_data["binned_face"].astype("category")

## Feature Dictionary

In [19]:
feature_dict = {
    "target": ["qx_log_raw"],
    "weight": ["amount_exposed"],
    "passthrough": ["attained_age", "duration", "observation_year"],
    "ordinal": [
        "sex",
        "smoker_status",
    ],
    "ohe": [
        "binned_face",
        "insurance_plan",
        "class_enh",
    ],
    "nominal": [],
}

feature_dict_vbt = {
    "target": ["qx_log_raw"],
    "weight": ["amount_exposed"],
    "passthrough": ["attained_age", "capped_duration"],
    "ordinal": [
        "sex",
        "smoker_status",
    ],
    "ohe": [],
    "nominal": [],
}

## Model Results Dictionary

In [20]:
metric_cols = ["ae", "smape", "r2_score", "root_mean_squared_error", "aic", "shape"]
model_results = metrics.ModelResults(metrics=metric_cols)

### VBT15

In [21]:
model_name = "vbt15"

In [22]:
scorecard = model_results.get_scorecard(
    y_true_train=model_data["death_claim_amount"],
    y_pred_train=model_data[f"exp_amt_{model_name}"],
    weights_train=None,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=None,
    model_params=None,
    scorecard=scorecard,
    importance=None,
)

 2025-05-13 00:16:40 | morai.forecast.metrics | INFO     | Adding model 'vbt15' 


# Forecasting Models

## Neural

**Feature Preprocessing:**
  - NN - using one-hot enconding (ohe) as the model needs the categories to be diferentiated and not ordinal or nominal. This only applies to non-binary features.
  - Scaling should be considered to ensure the features aren't overly weighted by larger values.

**Model Characteristics**
  - NN is a complex type of model

In [23]:
model_name = "neural"

In [24]:
preprocess_dict = preprocessors.preprocess_data(
    model_data,
    feature_dict=feature_dict_vbt,
    standardize=True,
)

 2025-05-13 00:16:44 | morai.forecast.preprocessors | INFO     | model target: ['qx_log_raw'] 
 2025-05-13 00:16:44 | morai.forecast.preprocessors | INFO     | model weights: ['amount_exposed'] 
 2025-05-13 00:16:44 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['capped_duration', 'attained_age'] 
 2025-05-13 00:16:44 | morai.forecast.preprocessors | INFO     | ordinal - ordinal encoded: ['sex', 'smoker_status'] 
 2025-05-13 00:16:45 | morai.forecast.preprocessors | INFO     | standardizing the data with StandardScaler 


In [25]:
X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]
md_encoded = preprocess_dict["md_encoded"]
model_features = preprocess_dict["model_features"]

X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(
    X, y, weights, random_state=0, test_size=0.2
)

In [28]:
model = neural.Neural()

In [29]:
model.fit(
    X_train=X_train, y_train=y_train, weights_train=None, epochs=100, lr=0.0001
)

 2025-05-13 00:17:23 | morai.models.neural | INFO     | training model None 
epochs: `100`,
optimizer: `Adam`,
criterion: `MSELoss`,
learning rate: `0.0001` 
Epoch [10/100], Loss: 0.3487
Epoch [20/100], Loss: 0.3459
Epoch [30/100], Loss: 0.3432
Epoch [40/100], Loss: 0.3405
Epoch [50/100], Loss: 0.3379
Epoch [60/100], Loss: 0.3353
Epoch [70/100], Loss: 0.3328
Epoch [80/100], Loss: 0.3303
Epoch [90/100], Loss: 0.3279
Epoch [100/100], Loss: 0.3255


In [30]:
predictions = np.exp(model.predict(X)) - 1

In [31]:
model_data = experience.calc_qx_exp_ae(
    model_data=model_data,
    predictions=predictions,
    model_name=model_name,
    exposure_col="amount_exposed",
    actual_col="death_claim_amount",
)

In [32]:
charters.compare_rates(
    model_data[model_data["insurance_plan"].isin(["UL"])],
    x_axis="attained_age",
    rates=["qx_raw", "qx_vbt15", "qx_neural"],
    weights=["amount_exposed"],
    secondary="death_count",
    y_log=False,
)

 2025-05-13 00:17:45 | morai.experience.charters | INFO     | The weights list is 1 long and should be 3 long. Using the first weight for all weights. 


# Reload

In [24]:
import importlib

In [25]:
importlib.reload(neural)

<module 'morai.integrations.neural' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\integrations\\neural.py'>